# Decoding Corridor Depth from Treadmill and Motor Kinematics

This notebook decodes corridor depth using only treadmill/motor kinematics—running speed (RS) and optic flow (OF)—without utilizing neuronal activity. It runs across V1 motor sessions for projects `ccyp_l5_3d_vision` and `colasa_3d-vision_revisions`.

According to virtual reality physics, the relationship between running speed, optic flow, and depth is given by:
$$OF = C \cdot \frac{RS}{\text{depth}}$$
Taking the natural log of both sides, this translates to the log-linear relationship:
$$\log(\text{depth}) = \log(C) + \log(RS) - \log(OF)$$

Thus, linear regression of $\log(\text{depth})$ against $\log(RS)$ and $\log(OF)$ combined should achieve a cross-validated $R^2 \approx 1.0$ (perfect decoding in theory).

We decode from:
1. Optic Flow (OF) only
2. Running Speed (RS) only
3. Both RS and OF combined

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import flexiznam as flz
import numpy as np
import pandas as pd
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from tqdm.notebook import tqdm

from cottage_analysis.summary_analysis import get_session_list, load_project_subsets
from cottage_analysis.analysis import treadmill, spheres

# Set style
sns.set_theme(style="ticks", context="talk")
font_path = '/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'


SESSIONS_TO_EXCLUDE = {
    "PZAG22.1b_S20260220": "1000 more frames than triggers in the treadmill recording, recording was wrongly started"
}

In [ ]:
def safe_log(arr):
    """
    Safely apply log-transform by setting non-positive values to NaN.
    """
    target = np.array(arr, dtype=float)
    invalid = target <= 0
    target[invalid] = np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.log(target)

def load_session_trials(sess, project, flexilims_session, cut_treadmill=True):
    """
    Load and synchronize treadmill or spheres recording for a session.
    """
    if ("PZAH6.4b" in sess) or ("PZAG3.4f" in sess):
        photodiode_protocol = 2
    else:
        photodiode_protocol = 5

    exp_session = flz.get_entity(
        datatype="session", name=sess, flexilims_session=flexilims_session
    )
    recordings = flz.get_entities(
        datatype="recording",
        origin_id=exp_session["id"],
        flexilims_session=flexilims_session,
    )
    is_treadmill = False
    if "protocol" in recordings.columns:
        is_treadmill = (
            recordings["protocol"]
            .str.contains("SpheresTubeMotor|Treadmill", na=False)
            .any()
        )
    if not is_treadmill:
        is_treadmill = (
            recordings["name"]
            .str.contains("SpheresTubeMotor|Treadmill", na=False)
            .any()
        )

    filter_datasets = {"anatomical_only": 3, "ast_neuropil": False}
    annotated_filter = dict(filter_datasets)
    annotated_filter["annotated"] = True

    def _do_sync(filt):
        if is_treadmill:
            acc_time = 0.13 if cut_treadmill else None
            return treadmill.sync_all_recordings(
                session_name=sess,
                flexilims_session=flexilims_session,
                project=project,
                filter_datasets=filt,
                recording_type="two_photon",
                photodiode_protocol=photodiode_protocol,
                return_volumes=True,
                acceleration_time=acc_time,
                cut_trial_end=None,
                trial_duration=None,
            )
        else:
            return spheres.sync_all_recordings(
                session_name=sess,
                flexilims_session=flexilims_session,
                project=project,
                filter_datasets=filt,
                recording_type="two_photon",
                protocol_base="SpheresPermTubeReward",
                photodiode_protocol=photodiode_protocol,
                return_volumes=True,
            )

    try:
        vs_df, trials_df = _do_sync(annotated_filter)
    except FileNotFoundError:
        vs_df, trials_df = _do_sync(filter_datasets)

    return trials_df

In [ ]:
def decode_depth_from_features(trials_df, feature_cols, k_folds=5, random_state=42, return_details=False):
    """
    Perform k-fold cross-validation at the trial level to decode depth
    using linear regression on log-transformed variables.
    If return_details is True, also collects frame-level errors and feature arrays.
    """
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)
    trial_indices = np.arange(len(trials_df))
    
    y_test_all = []
    y_pred_all = []
    details = []
    
    for train_idx, test_idx in kf.split(trial_indices):
        train_trials = trials_df.iloc[train_idx]
        test_trials = trials_df.iloc[test_idx]
        
        # Build training set (concatenate frames across trials)
        X_train_list = []
        y_train_list = []
        for _, row in train_trials.iterrows():
            n_frames = len(row["RS_stim"])
            y_val = np.repeat(row["depth"], n_frames)
            
            feats = []
            for col in feature_cols:
                feats.append(row[col])
            feats = np.column_stack(feats)
            
            X_train_list.append(feats)
            y_train_list.append(y_val)
            
        X_train = np.vstack(X_train_list)
        y_train = np.hstack(y_train_list)
        
        # Log-transform
        X_train_log = safe_log(X_train)
        y_train_log = safe_log(y_train)
        
        # Drop NaNs
        valid_train = ~np.isnan(y_train_log) & ~np.any(np.isnan(X_train_log), axis=1)
        X_train_log = X_train_log[valid_train]
        y_train_log = y_train_log[valid_train]
        
        if len(y_train_log) == 0:
            continue
            
        # Fit model
        model = LinearRegression()
        model.fit(X_train_log, y_train_log)
        
        # Build test set
        for _, row in test_trials.iterrows():
            n_frames = len(row["RS_stim"])
            y_val = np.repeat(row["depth"], n_frames)
            
            feats = []
            for col in feature_cols:
                feats.append(row[col])
            feats = np.column_stack(feats)
            
            X_test_log = safe_log(feats)
            y_test_log = safe_log(y_val)
            rs_log = safe_log(np.column_stack([row["RS_stim"]]))
            
            valid_test = ~np.isnan(y_test_log) & ~np.any(np.isnan(X_test_log), axis=1)
            if return_details:
                valid_test = valid_test & ~np.any(np.isnan(rs_log), axis=1)
            
            if np.sum(valid_test) > 0:
                pred = model.predict(X_test_log[valid_test])
                y_pred_all.append(pred)
                y_test_all.append(y_test_log[valid_test])
                
                if return_details:
                    for p, t, r in zip(pred, y_test_log[valid_test], rs_log[valid_test].flatten()):
                        details.append({
                            "error": t - p,
                            "abs_error": abs(t - p),
                            "true_log_depth": t,
                            "true_depth": np.exp(t),
                            "rs_log": r,
                            "rs": np.exp(r)
                        })
                
    if len(y_test_all) == 0:
        if return_details:
            return np.nan, []
        return np.nan
        
    y_test_all = np.concatenate(y_test_all)
    y_pred_all = np.concatenate(y_pred_all)
    
    r2 = r2_score(y_test_all, y_pred_all)
    if return_details:
        return r2, details
    return r2

In [ ]:
projects = ["ccyp_l5_3d_vision", "colasa_3d-vision_revisions"]
results = []
all_errors = []
cut_treadmill = True

for project in projects:
    flexilims_session = flz.get_flexilims_session(project_id=project)
    
    # Load project subsets to identify the exact sessions present in summary plots
    df_subsets = load_project_subsets(
        project,
        session_to_exclude=SESSIONS_TO_EXCLUDE.keys(),
        cut_treadmill=cut_treadmill
    )
    sessions = sorted(df_subsets["session"].unique())
    project_sessions = flz.get_entities("session", flexilims_session=flexilims_session)
    
    print(f"\nProcessing {len(sessions)} sessions in project {project}...")
    for sess in tqdm(sessions):
        if sess in SESSIONS_TO_EXCLUDE:
            continue
        
        if sess in project_sessions.index:
            nominal_depth = project_sessions.loc[sess, "nominal_depth"]
            if isinstance(nominal_depth, (list, np.ndarray, pd.Series)):
                nominal_depth = np.mean(nominal_depth)
        else: 
            continue
            
        try:
            trials_df = load_session_trials(sess, project, flexilims_session, cut_treadmill=cut_treadmill)
            
            # Keep only closed-loop trials
            trials_df = trials_df[trials_df.closed_loop == 1].copy()
            if len(trials_df) == 0:
                print(f"Session {sess}: no closed loop trials found. Skipping.")
                continue
                
            # Perform decoding (combining both computation and frame-level detail collection into 1 pass)
            r2_of, of_details = decode_depth_from_features(trials_df, ["OF_stim"], return_details=True)
            r2_rs = decode_depth_from_features(trials_df, ["RS_stim"])
            r2_both = decode_depth_from_features(trials_df, ["RS_stim", "OF_stim"])
            
            results.append({
                "session": sess,
                "project": project,
                "nominal_depth": nominal_depth,
                "r2_OF": r2_of,
                "r2_RS": r2_rs,
                "r2_both": r2_both
            })
            
            for d in of_details:
                d["session"] = sess
                d["project"] = project
                all_errors.append(d)
            
        except Exception as e:
            print(f"Error processing session {sess}: {e}")

results_df = pd.DataFrame(results)
errors_df = pd.DataFrame(all_errors)

In [ ]:
print("\n=== Summary of R2 Scores ===")
display(results_df.head(20))
results_df.to_parquet("depth_decoding_kinematics_results.parquet", index=False)

In [ ]:
# Melt the DataFrame for plotting
df_melt = pd.melt(
    results_df, 
    id_vars=["session", "project", "nominal_depth"], 
    value_vars=["r2_OF", "r2_RS", "r2_both"],
    var_name="decoder", 
    value_name="r2"
)

# Clean up names for plot
df_melt["decoder"] = df_melt["decoder"].map({
    "r2_OF": "Optic Flow (OF) only",
    "r2_RS": "Running Speed (RS) only",
    "r2_both": "Both (RS + OF)"
})

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_melt, x="decoder", y="r2", palette="Set2", showfliers=False)
sns.stripplot(data=df_melt, x="decoder", y="r2", color="black", alpha=0.6, jitter=0.2, size=6)

plt.title("Decoding Depth from Treadmill Kinematics (No Neurons)")
plt.ylabel("Cross-validated $R^2$")
plt.xlabel("Decoder Type")
plt.ylim(-0.1, 1.1)
sns.despine()
plt.tight_layout()
plt.savefig("depth_decoding_kinematics_plot.png", dpi=150)
plt.show()

## Error Analysis for the Optic Flow (OF) Only Decoder

Since the physical relationship is $\log(\text{depth}) = \log(C) + \log(RS) - \log(OF)$, predicting depth from optic flow alone ignores the independent variation of running speed ($RS$). Consequently, the prediction error for the OF-only decoder should be highly structured and depend on both the true depth and the running speed ($RS$).

Here we analyze the frame-by-frame cross-validated prediction errors for the OF-only decoder collected during our single pass through the sessions.

In [ ]:
from scipy.stats import pearsonr

corr_depth_err, p_depth_err = pearsonr(errors_df["true_log_depth"], errors_df["error"])
corr_depth_abs, p_depth_abs = pearsonr(errors_df["true_log_depth"], errors_df["abs_error"])

corr_rs_err, p_rs_err = pearsonr(errors_df["rs_log"], errors_df["error"])
corr_rs_abs, p_rs_abs = pearsonr(errors_df["rs_log"], errors_df["abs_error"])

print("--- Correlation with Prediction Error (true_log_depth - pred_log_depth) ---")
print(f"True Log Depth vs Error: r = {corr_depth_err:.4f} (p = {p_depth_err:.2e})")
print(f"Log RS vs Error:         r = {corr_rs_err:.4f} (p = {p_rs_err:.2e})")
print("\n--- Correlation with Absolute Prediction Error (|Error|) ---")
print(f"True Log Depth vs Absolute Error: r = {corr_depth_abs:.4f} (p = {p_depth_abs:.2e})")
print(f"Log RS vs Absolute Error:         r = {corr_rs_abs:.4f} (p = {p_rs_abs:.2e})")

In [ ]:
# Subsample for plotting
sample_df = errors_df.sample(n=min(len(errors_df), 10000), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=False)

# Error vs True Depth
sns.regplot(data=sample_df, x="true_depth", y="error", ax=axes[0], palette="Blues")
axes[0].set_title("OF-Only Prediction Error vs True Depth")
axes[0].set_xlabel("True Corridor Depth (cm)")
axes[0].set_ylabel("Prediction Error (Log Depth Diff)")
axes[0].axhline(0, color="red", linestyle="--", alpha=0.7)

# Error vs Running Speed
sns.regplot(data=sample_df, x="rs_log", y="error", ax=axes[1], 
            scatter_kws={"alpha": 0.2, "color": "purple"}, line_kws={"color": "red"})
axes[1].set_title("OF-Only Prediction Error vs Log RS")
axes[1].set_xlabel("Log Running Speed ($\\log(RS)$)")
axes[1].set_ylabel("Prediction Error (Log Depth Diff)")
axes[1].axhline(0, color="red", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.savefig("of_decoder_error_analysis.png", dpi=150)
plt.show()